In [5]:
import duckdb

In [7]:
con = duckdb.connect(database='dados_duckdb.db', read_only=False)

In [8]:
df = con.execute("""
                 SELECT * 
                 FROM (
                    SELECT *, ROW_NUMBER() OVER (PARTITION BY NATBR ORDER BY data_ingestao DESC) AS row
                    FROM bronze_z0019
                    WHERE data_ingestao >= '2025-01-11'
                ) WHERE row = 1
                 """).fetchdf()
df.head(10)

,NATBR,MAKTX,WERKS,MAINS,LABST,nome_arquivo,data_ingestao,row
0,1001,PARAFUSO,BT10,100,100,z0019_1.csv,2025-10-03 11:02:13.346356,1
1,1003,PREGO,BT10,100,60,z0019_2.csv,2025-10-03 11:05:44.295367,1
2,1002,MARTELO,BT50,100,1500,z0019_1.csv,2025-10-03 11:02:13.346356,1
3,1005,MACHADO,BT50,100,100,z0019_2.csv,2025-10-03 11:05:44.295367,1
4,1004,SERRA,BT50,100,200,z0019_2.csv,2025-10-03 11:05:44.295367,1


In [9]:
df_final = df.drop(columns=['nome_arquivo', 'data_ingestao', 'row'])
df_final = df_final.rename(columns={"NATBR":"id"})
df_final = df_final.rename(columns={"MAKTX":"nm_produto"})
df_final = df_final.rename(columns={"WERKS":"id_categoria"})
df_final = df_final.rename(columns={"MAINS":"id_fornecedor"})
df_final = df_final.rename(columns={"LABST":"vl_preco"})
df_final.head(10)

,id,nm_produto,id_categoria,id_fornecedor,vl_preco
0,1001,PARAFUSO,BT10,100,100
1,1003,PREGO,BT10,100,60
2,1002,MARTELO,BT50,100,1500
3,1005,MACHADO,BT50,100,100
4,1004,SERRA,BT50,100,200


In [10]:
df_final.dtypes

id               object
nm_produto       object
id_categoria     object
id_fornecedor    object
vl_preco         object
dtype: object

In [11]:
df2 = df_final
df2 = df2.astype(
    {
        'id': int,
        'nm_produto': str,
        'id_categoria': str,
        'id_fornecedor': int,
        'vl_preco': float
    }
)

df2.dtypes

id                 int64
nm_produto        object
id_categoria      object
id_fornecedor      int64
vl_preco         float64
dtype: object

In [12]:
con.execute("""
CREATE TABLE IF NOT EXISTS produtos (
            id BIGINT,
            nm_produto TEXT,
            id_categoria TEXT,
            id_fornecedor BIGINT,
            vl_preco FLOAT
            )
""")

In [13]:
df2.head(10)

,id,nm_produto,id_categoria,id_fornecedor,vl_preco
0,1001,PARAFUSO,BT10,100,100.0
1,1003,PREGO,BT10,100,60.0
2,1002,MARTELO,BT50,100,1500.0
3,1005,MACHADO,BT50,100,100.0
4,1004,SERRA,BT50,100,200.0


In [14]:
con.execute("INSERT INTO produtos SELECT * from df2")

In [15]:
df_resultado = con.execute("select * from produtos").fetchdf()
df_resultado.head(10)

,id,nm_produto,id_categoria,id_fornecedor,vl_preco
0,1001,PARAFUSO,BT10,100,100.0
1,1003,PREGO,BT10,100,60.0
2,1002,MARTELO,BT50,100,1500.0
3,1005,MACHADO,BT50,100,100.0
4,1004,SERRA,BT50,100,200.0


In [16]:
con.close()